In [1]:
import requests
import zipfile
import io
import pandas as pd

def download_unzip_and_read_excel(url, output_folder='.'):
    """
    Download a zip file from URL, unzip it, and read the Excel file.
    
    Args:
        url (str): URL of the zip file to download
        output_folder (str): Folder to save the extracted files (default: current directory)
    
    Returns:
        pandas.DataFrame: Contents of the Excel file
    """
    try:
        print(f"Downloading zip file from {url}...")
        # Download the zip file
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Unzip the file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Find the Excel file in the zip (assuming there's only one)
            excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls', '.csv'))]
            
            if not excel_files:
                raise ValueError("No Excel/CSV file found in the zip archive")
            
            # We'll use the first Excel file found
            excel_filename = excel_files[0]
            print(f"Found file in zip: {excel_filename}")
            
            # Extract the file (either to memory or disk)
            if output_folder:
                print(f"Extracting to {output_folder}...")
                zip_ref.extractall(output_folder)
                file_path = f"{output_folder}/{excel_filename}"
                print(f"Reading Excel file from {file_path}...")
                
                # Read the file based on extension
                if excel_filename.endswith('.csv'):
                    df = pd.read_csv(file_path)
                else:  # .xlsx or .xls
                    df = pd.read_excel(file_path)
            else:
                # Extract to memory
                with zip_ref.open(excel_filename) as file:
                    print("Reading Excel file from memory...")
                    if excel_filename.endswith('.csv'):
                        df = pd.read_csv(file)
                    else:  # .xlsx or .xls
                        df = pd.read_excel(file)
            
            print("Successfully read the Excel file!")
            return df
    
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip file")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def download_unzip_and_read_excel_income(url, output_folder='.',name_sheet='ENSEMBLE'):
    """
    Download a zip file from URL, unzip it, and read the Excel file.
    
    Args:
        url (str): URL of the zip file to download
        output_folder (str): Folder to save the extracted files (default: current directory)
    
    Returns:
        pandas.DataFrame: Contents of the Excel file
    """
    try:
        print(f"Downloading zip file from {url}...")
        # Download the zip file
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Unzip the file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Find the Excel file in the zip (assuming there's only one)
            excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls', '.csv'))]
            
            if not excel_files:
                raise ValueError("No Excel/CSV file found in the zip archive")
            
            # We'll use the first Excel file found
            excel_filename = [f for f in excel_files if "DISP_COM" in f][0]
            print(f"Found file in zip: {excel_filename}")
            
            # Extract the file (either to memory or disk)
            if output_folder:
                print(f"Extracting to {output_folder}...")
                zip_ref.extractall(output_folder)
                file_path = f"{output_folder}/{excel_filename}"
                print(f"Reading Excel file from {file_path}...")
                
                # Read the file based on extension
                if excel_filename.endswith('.csv'):
                    df = pd.read_csv(file_path,error_bad_lines=False,sep=";")
                else:  # .xlsx or .xls
                    try:
                        df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5)
                    except:
                        df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5,engine='openpyxl')
            else:
                # Extract to memory
                with zip_ref.open(excel_filename) as file:
                    print("Reading Excel file from memory...")
                    if excel_filename.endswith('.csv'):
                        df = pd.read_csv(file,error_bad_lines=False,sep=";")
                    else:  # .xlsx or .xls
                        try:
                            df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5)
                        except:
                            df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5,engine='openpyxl')
            
            print("Successfully read the Excel file!")
            return df
    
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip file")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## Données vote législatives

In [ ]:
# URL of the zip file
full_df = pd.DataFrame()
years = [2022, 2017, 2012, 2007, 2002]
for year in years:

    zip_url = f"https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg{year}_csv.zip"

    # Download, unzip and read the Excel file
    data_frame = download_unzip_and_read_excel(zip_url)
    common_columns = ['dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits','votants', 'exprimes','pvoteG', 'pvoteCG','pvoteC', 'pvoteCD', 'pvoteD', 'pvoteTG', 'pvoteTD', 'pvoteGCG','pvoteDCD']
    data_frame = data_frame[common_columns]
    data_frame['Year']  = year
    full_df = full_df.append(data_frame, ignore_index=True)
    print(f"Data for year {year} added to the DataFrame.")

full_df = full_df.set_index(['Year', 'dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits', 'votants', 'exprimes']).stack().reset_index()
full_df.columns = ['Year', 'dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits', 'votants', 'exprimes', 'vote_type', 'share']
full_df['vote_type'] = full_df['vote_type'].apply(lambda x: x.replace('pvote', ''))
full_df['absention'] = 1 - full_df['votants'] / full_df['inscrits']



Found file in zip: leg2022_csv/leg2022comm.csv
Extracting to ....
Reading Excel file from ./leg2022_csv/leg2022comm.csv...


c:\Users\mehdi\anaconda3\envs\GEO\lib\site-packages\IPython\core\interactiveshell.py:3263: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


Successfully read the Excel file!
Data for year 2022 added to the DataFrame.
Found file in zip: leg2017_csv/leg2017comm.csv
Extracting to ....
Reading Excel file from ./leg2017_csv/leg2017comm.csv...
Successfully read the Excel file!
Data for year 2017 added to the DataFrame.
Found file in zip: leg2012_csv/leg2012comm.csv
Extracting to ....
Reading Excel file from ./leg2012_csv/leg2012comm.csv...
Successfully read the Excel file!
Data for year 2012 added to the DataFrame.
Found file in zip: leg2007_csv/leg2007comm.csv
Extracting to ....
Reading Excel file from ./leg2007_csv/leg2007comm.csv...
Successfully read the Excel file!
Data for year 2007 added to the DataFrame.
Found file in zip: leg2002_csv/leg2002comm.csv
Extracting to ....
Reading Excel file from ./leg2002_csv/leg2002comm.csv...
Successfully read the Excel file!
Data for year 2002 added to the DataFrame.


[Data 2021](https://www.insee.fr/fr/statistiques/7756855?sommaire=7756859&q=Dispositif+Fichier+localis%C3%A9%20social+et+fiscal+(Filosofi)), [Data 2017](https://www.insee.fr/fr/statistiques/4291712#consulter),[Data 2012](https://www.insee.fr/fr/statistiques/2043745)

In [ ]:
urls = ["https://www.insee.fr/fr/statistiques/fichier/2043745/indic-struct-distrib-revenu-communes-2012.zip",
        "https://www.insee.fr/fr/statistiques/fichier/4291712/indic-struct-distrib-revenu-2017-COMMUNES.zip",
        "https://www.insee.fr/fr/statistiques/fichier/7756855/indic-struct-distrib-revenu-2021-COMMUNES_csv.zip"]
years = [2012,2017, 2021]
full_rev = pd.DataFrame()
for i in range(len(urls)):
    zip_url = urls[i]
    rev = download_unzip_and_read_excel_income(zip_url, name_sheet='ENSEMBLE')
    year = years[i]
    rev = rev[['CODGEO','NBMEN%s'%str(year)[2:],'NBPERS%s'%str(year)[2:],'NBUC%s'%str(year)[2:],
                 'Q2%s'%str(year)[2:],'GI%s'%str(year)[2:]]] 
    rev.columns = ['CODGEO','Menages','Population','UC','Med_Niveau_vie','Gini']
    rev['Year'] = str(year)
    full_rev = full_rev.append(rev, ignore_index=True)


Found file in zip: indic-struct-distrib-revenu-communes-2012/FILO_DISP_COM.xls
Extracting to ....
Reading Excel file from ./indic-struct-distrib-revenu-communes-2012/FILO_DISP_COM.xls...
Successfully read the Excel file!


In [9]:
rev

,CODGEO,NBMEN21,NBPERS21,NBUC21,Q121,Q221,Q321,Q3_Q1,D121,D221,...,OPR6PTSA21,OPR6PCHO21,OPR6PBEN21,OPR6PPEN21,OPR6PPAT21,OPR6PPSOC21,OPR6PPFAM21,OPR6PPMINI21,OPR6PPLOGT21,OPR6PIMPOT21
0,1001,346,895,"590,8",s,25820,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
1,1002,115,266,"181,0",s,24480,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
2,1004,6855,15092,"10398,2",15800,21660,28430,12630,11890,14640,...,s,s,s,s,s,s,s,s,s,s
3,1005,800,2028,"1329,7",20010,24610,31180,11170,15560,18980,...,s,s,s,s,s,s,s,s,s,s
4,1006,51,107,"76,6",s,24210,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34924,97420,8593,23979,"15260,6",11970,17720,26400,14430,9060,11030,...,s,s,s,s,s,s,s,s,s,s
34925,97421,2472,6847,"4379,7",9860,13280,19240,9380,6810,9150,...,s,s,s,s,s,s,s,s,s,s
34926,97422,31239,79757,"52192,7",11440,16560,25020,13580,8660,10670,...,s,s,s,s,s,s,s,s,s,s
34927,97423,2505,7019,"4507,7",11480,16680,24240,12750,8570,10760,...,s,s,s,s,s,s,s,s,s,s
